# Technical Analysis with SMA & MACD crossover

This notebook draws stock data from Yahoo Finance and uses Technical indicators from pandas to analyse entry points

In [171]:
# Import Libraries
from datetime import datetime, timedelta
import pytz
import numpy as np
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [172]:
# User-defined parameters
SYMBOL           = "SPY"       #Enter the stock ticker symbol here
INTERVAL         = "1d"         # 1-hour candles
SMA_LEN          = 200          # SMA length for trend direction 
BACKCANDLES_PREV = 3            # number of previous candles to consider for swing trading
END = (datetime.now(pytz.utc)).date().isoformat() 
START = (datetime.now(pytz.utc) - timedelta(days=3000)).date().isoformat()   # 2-3 year worth of data

In [173]:
# Fetch historical data using yfinance
def fetch_data(symbol: str, start: str, end: str, interval: str) -> pd.DataFrame:
    df = yf.download(symbol, start=start, end=end, interval=interval,
                     auto_adjust=True, progress=False, threads=False)

    if isinstance(df.columns, pd.MultiIndex):
        try:
            df = df.xs(symbol, axis=1, level=1)
        except KeyError:
            possible = [lev for lev in df.columns.levels[1]]
            raise KeyError(f"Symbol '{symbol}' not found in MultiIndex columns. "
                           f"Available: {possible}")
    else:
        pass

    # Ensure standardized column names
    df.columns = [c.title() for c in df.columns]
    return df.dropna()

In [174]:
# SMA trend signal
def sma_trend_signal(df: pd.DataFrame, i: int, backcandles_prev: int) -> int:
    """
    Return:
      1  if ALL of [i-backcandles_prev .. i] have Open>SMA and Close>SMA  (uptrend)
     -1  if ALL of [i-backcandles_prev .. i] have Open<SMA and Close<SMA  (downtrend)
      0  otherwise
    """
    if i < backcandles_prev:
        return 0
    if np.isnan(df["SMA200"].iloc[i]):
        return 0

    start = i - backcandles_prev
    seg = df.iloc[start:i+1]
    up   = ((seg["Open"] > seg["SMA200"]) & (seg["Close"] > seg["SMA200"])).all()
    down = ((seg["Open"] < seg["SMA200"]) & (seg["Close"] < seg["SMA200"])).all()
    return 1 if up else (-1 if down else 0)

In [175]:
# Build features
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Indicators
    out["SMA200"] = ta.sma(out["Close"], length=SMA_LEN)
    macd = ta.macd(out["Close"], fast=12, slow=26, signal=9)
    out = out.join(macd)

    # sma_signal via the function specified
    out["sma_signal"] = 0
    out["sma_signal"] = [sma_trend_signal(out, i, BACKCANDLES_PREV) for i in range(len(out))]

    # MACD cross logic with pullback confirmation via histogram
    macd_line = out["MACD_12_26_9"]
    macd_sig  = out["MACDs_12_26_9"]
    macd_hist = out["MACDh_12_26_9"]
    macd_line_prev = macd_line.shift(1)
    macd_sig_prev  = macd_sig.shift(1)
    # macd_hist_prev = macd_hist.shift(1)

    # Params
    hist_thresh = 4e-6
    hist_window = 3 
    # "Any of last 3 bars" condition via rolling extremes
    hist_below_win = macd_hist.rolling(hist_window, min_periods=hist_window).min() < -hist_thresh
    hist_above_win = macd_hist.rolling(hist_window, min_periods=hist_window).max() >  hist_thresh

    # --- Entries ---
    # Long: MACD line crosses ABOVE signal line while both are BELOW zero (bullish resumption)
    bull_cross_below0 = (
        hist_below_win &                          # pullback was deep enough in last 3 bars
        (macd_line_prev <= macd_sig_prev) &       # was not above on prior bar
        (macd_line > macd_sig) &                  # crosses above now
        (macd_line < 0) & (macd_sig < 0)          # cross happens below zero
    )

    # Short: MACD line crosses BELOW signal line while both are ABOVE zero (bearish resumption)
    bear_cross_above0 = (
        hist_above_win &                          # pullback was deep enough in last 3 bars
        (macd_line_prev >= macd_sig_prev) &       # was not below on prior bar
        (macd_line < macd_sig) &                  # crosses below now
        (macd_line > 0) & (macd_sig > 0)          # cross happens above zero
    )

    out["MACD_signal"] = 0
    out.loc[bull_cross_below0, "MACD_signal"] = 1
    out.loc[bear_cross_above0, "MACD_signal"] = -1

    # Precomputed combined signal
    out["pre_signal"] = 0
    out.loc[(out["sma_signal"] == 1) & (out["MACD_signal"] == 1), "pre_signal"] = 1
    out.loc[(out["sma_signal"] == -1) & (out["MACD_signal"] == -1), "pre_signal"] = -1
    
    # Drop early NaNs from indicators
    out = out.dropna().copy()
    return out

In [176]:
# Main execution
raw = fetch_data(SYMBOL, START, END, INTERVAL)
df  = build_features(raw)

# Plotting function
def plot_sma_macd(
    df: pd.DataFrame,
    sma_col: str = "SMA200",
    signal_col: str = "pre_signal",
    macd_col: str = "MACD_12_26_9",
    macds_col: str = "MACDs_12_26_9",
    macdh_col: str = "MACDh_12_26_9",
    title: str | None = None,
    start: str | None = None,
    end: str | None = None,
):
    # Checks
    required_ohlc = {"Open", "High", "Low", "Close"}
    missing = required_ohlc - set(df.columns)
    if missing:
        raise ValueError(f"Missing OHLC columns: {missing}")
    for col in [sma_col, signal_col, macd_col, macds_col, macdh_col]:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in df.")

    # Slice
    _df = df.loc[start:end].copy() if (start or end) else df.copy()
    if _df.empty:
        raise ValueError("Selected slice is empty. Check 'start'/'end'.")

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        row_heights=[0.68, 0.32],
        subplot_titles=(title or "Price / SMA / Signals", "MACD")
    )

    # Row 1: Candles
    fig.add_trace(
        go.Candlestick(
            x=_df.index, open=_df["Open"], high=_df["High"],
            low=_df["Low"], close=_df["Close"],
            name="Candles", showlegend=False
        ),
        row=1, col=1
    )

    # SMA
    fig.add_trace(
        go.Scatter(x=_df.index, y=_df[sma_col], mode="lines", name=sma_col),
        row=1, col=1
    )

    # Signals → purple triangles
    sig = _df[signal_col].fillna(0)
    hl_range = (_df["High"] - _df["Low"]).astype(float)
    # Use 'where' to substitute 0 ranges with a tiny fraction of price
    hl_range = hl_range.where(hl_range != 0, _df["Close"] * 0.001)
    offset = np.maximum(hl_range * 0.2, _df["Close"] * 0.0005)

    long_mask = sig.eq(1)
    if long_mask.any():
        long_y = (_df["Low"] - offset)[long_mask]
        fig.add_trace(
            go.Scatter(
                x=_df.index[long_mask], y=long_y,
                mode="markers",
                marker=dict(symbol="triangle-up", size=15, color="green"),
                name="Long (+1)"
            ),
            row=1, col=1
        )

    short_mask = sig.eq(-1)
    if short_mask.any():
        short_y = (_df["High"] + offset)[short_mask]
        fig.add_trace(
            go.Scatter(
                x=_df.index[short_mask], y=short_y,
                mode="markers",
                marker=dict(symbol="triangle-down", size=15, color="red"),
                name="Short (-1)"
            ),
            row=1, col=1
        )

    # Row 2: MACD
    fig.add_trace(go.Bar(x=_df.index, y=_df[macdh_col], name="MACD Hist", opacity=0.5), row=2, col=1)
    fig.add_trace(go.Scatter(x=_df.index, y=np.zeros(len(_df)), mode="lines",
                             line=dict(width=1, dash="dash"), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=_df.index, y=_df[macd_col],  mode="lines", name="MACD"),  row=2, col=1)
    fig.add_trace(go.Scatter(x=_df.index, y=_df[macds_col], mode="lines", name="Signal"), row=2, col=1)

    fig.update_layout(
        title=title or "SMA & Signals with MACD",
        xaxis_rangeslider_visible=False,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=40, r=20, t=70, b=40),
        height=800
    )
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="MACD",  row=2, col=1)
    return fig


fig = plot_sma_macd(
    df,
    sma_col="SMA200",
    signal_col="pre_signal",
    macd_col="MACD_12_26_9",
    macds_col="MACDs_12_26_9",
    macdh_col="MACDh_12_26_9",
    title=SYMBOL + " - SMA & MACD Signals"
)
fig.show()

In [177]:
df

,Close,High,Low,Open,Volume,SMA200,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9,sma_signal,MACD_signal,pre_signal
Date,,,,,,,,,,,,
2018-10-10,248.831482,256.529804,248.455970,256.458260,214731000,245.047577,-0.358252,-1.129343,0.771091,0,0,0
2018-10-11,243.350586,249.367945,241.732217,247.740654,274840500,245.085262,-1.438548,-1.767711,0.329163,0,0,0
2018-10-12,246.730301,247.749575,243.529364,247.463452,183186500,245.139271,-1.998933,-1.862476,-0.136456,0,0,0
2018-10-15,245.344467,247.704937,245.255050,246.372691,102263700,245.183923,-2.525751,-1.911436,-0.614315,0,0,0
2018-10-16,250.709229,251.084768,246.837732,247.311611,118255800,245.259857,-2.481760,-1.493956,-0.987804,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-05,681.309998,685.530029,675.609985,682.080017,106606500,653.714477,-1.112541,-0.564771,-0.547770,1,0,0
2026-03-06,672.380005,676.109985,669.760010,673.409973,100687000,654.127805,-2.064841,-1.213657,-0.851184,1,0,0
2026-03-09,678.270020,679.919983,662.390015,666.390015,102667700,654.580498,-2.317556,-1.173097,-1.144459,1,0,0
